# Task-trained RNN — geometry & dynamics of a *learned* network

The ring-attractor notebook analysed a network whose structure we **built in**.
Here we do the opposite: **train** a recurrent network on a cognitive task and then
ask the library what geometry it *discovered*. This exercises the **pullback /
dynamics** half of `neuralgeom` (Jacobians, spectra, fixed points, the state-space
pullback metric) and then bridges back to the **subspace lens** on the same
network — the payoff of merging the two toolkits.

**Task: evidence integration.** On each trial two input channels ("A" and "B")
emit brief pulses over a variable window; the network must report which channel
fired more. Solving this the standard way requires the network to **accumulate**
the difference and hold it — the textbook mechanism is a **line attractor**: a
1-D continuum of near-stable states along which the accumulated evidence is
stored. So going in, we expect to be able to describe:
- an approximately **one-dimensional slow manifold** (line attractor) in state space;
- **recurrent Jacobian eigenvalues near |λ| = 1** along it (long memory /
  effective time constants far exceeding the single-unit τ);
- **slow points** strung out along that line;
- an accumulation axis that aligns with the **readout** direction.

Nothing about the structure is hand-coded — it has to emerge from training.
Because training is stochastic, exact numbers vary run to run; the *shapes* are
the point.

In [ ]:
%matplotlib inline
import numpy as np, torch
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.axisbelow": True})
torch.manual_seed(0); np.random.seed(0)
import neuralgeom as ng
print("neuralgeom", ng.__version__, "| torch", torch.__version__)

## 1. Build and train the network

A leaky continuous-time **vanilla RNN**, `h ← (1−α)h + α·tanh(W_rec h + W_in x + b)`
with `α = dt/τ`, trained with Adam on masked cross-entropy. We add a small L2
penalty on the rates ("metabolic cost"), which encourages robust attractors
rather than fragile transients.

In [ ]:
from neuralgeom.tasks import make_task, make_model, train, evaluate, run_trials

task  = make_task("evidence_integration", dt=20, seed=0)     # dt = 20 ms
model = make_model("vanilla", task.spec, hidden_size=128, tau=100.0)
print(task.spec)

hist = []
train(model, task, steps=2500, batch_size=64, lr=2e-3, l2_rate=1e-3,
      log_every=100, on_log=lambda step, d: hist.append((step, d)), verbose=False)
acc = float(evaluate(task=task, model=model))
print(f"final held-out accuracy ≈ {acc:.2f}")

steps = [s for s, _ in hist]
def series(key):
    return [d.get(key, np.nan) for _, d in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(steps, series("loss"), color="C0"); ax[0].set(title="training loss", xlabel="step")
ax[1].plot(steps, series("acc"), color="C2"); ax[1].set(title="training accuracy", xlabel="step", ylim=(0,1.02))
fig.tight_layout(); display(fig); plt.close(fig)

## 2. Behaviour and example trials

Run a batch of fresh trials. `run_trials` returns the `TrialBatch` (with per-trial
metadata like the true `evidence = n_A − n_B` and `choice`), the output logits, and
the full hidden-state tensor `(trials, time, N)`. We look at a couple of example
trials and the **psychometric curve** — P(choose A) as a function of the evidence.

In [ ]:
batch, outputs, hidden = run_trials(model, task, batch_size=400)
outputs = outputs.detach(); hidden = hidden.detach()
ev = np.asarray(batch.meta["evidence"])
# model choice = argmax logit at the last timestep (labels: 0 fixate,1 A,2 B)
choice = outputs[:, -1, :].argmax(-1).numpy()
chose_A = (choice == 1).astype(float)

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for j in range(3):
    tr = np.argsort(ev)[[2, len(ev)//2, -3][j]]
    ax[j].plot(batch.inputs[tr, :, 1], label="pulses A", color="C0")
    ax[j].plot(batch.inputs[tr, :, 2], label="pulses B", color="C1")
    ax[j].plot(outputs[tr, :, 1], "--", label="logit A", color="C0", alpha=0.7)
    ax[j].plot(outputs[tr, :, 2], "--", label="logit B", color="C1", alpha=0.7)
    ax[j].set(title=f"trial (evidence={ev[tr]:+.0f})", xlabel="time step")
    if j == 0: ax[j].legend(fontsize=7)
fig.tight_layout(); display(fig); plt.close(fig)

# psychometric: bin P(choose A) by evidence
bins = np.arange(ev.min(), ev.max()+2) - 0.5
idx = np.digitize(ev, bins)
xs, ps = [], []
for b in np.unique(idx):
    m = idx == b
    if m.sum() >= 3:
        xs.append(ev[m].mean()); ps.append(chose_A[m].mean())
fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.plot(xs, ps, "-o", color="C3"); ax.axhline(0.5, color="0.6", ls=":"); ax.axvline(0, color="0.6", ls=":")
ax.set(title=f"psychometric (acc≈{acc:.2f})", xlabel="evidence  n_A − n_B", ylabel="P(choose A)", ylim=(-0.02,1.02))
fig.tight_layout(); display(fig); plt.close(fig)

## 3. Population geometry, coloured by the computed variable

Project the hidden states onto their top PCs and colour by the **accumulated
evidence**. If the network integrates along a line attractor, evidence should be
laid out **monotonically along one axis** of state space.

In [ ]:
H = hidden.reshape(-1, hidden.shape[-1]).numpy()
ev_full = np.repeat(ev, hidden.shape[1])
Hc = H - H.mean(0)
Uh, Sh, Vth = np.linalg.svd(Hc, full_matrices=False)
P = Hc @ Vth[:3].T
from neuralgeom.dynamics import participation_ratio
pr = participation_ratio(hidden.reshape(-1, hidden.shape[-1]))

fig = plt.figure(figsize=(13, 4.2))
ax0 = fig.add_subplot(1, 2, 1, projection="3d")
s = ax0.scatter(P[::5,0], P[::5,1], P[::5,2], c=ev_full[::5], cmap="coolwarm", s=5)
ax0.set(title="hidden states (PCA 3), coloured by evidence", xlabel="PC1", ylabel="PC2"); ax0.set_zlabel("PC3")
fig.colorbar(s, ax=ax0, label="evidence", shrink=0.6)
ax1 = fig.add_subplot(1, 2, 2)
# end-of-trial state vs evidence along PC1
endP = (hidden[:, -1, :].numpy() - H.mean(0)) @ Vth[0]
ax1.scatter(ev, endP, s=14, c="C0", alpha=0.6)
ax1.set(title=f"end-state PC1 vs evidence  (participation ratio ≈ {pr:.1f})",
        xlabel="evidence", ylabel="PC1 of final state")
fig.tight_layout(); display(fig); plt.close(fig)

Read the right-hand panel as the test: if the final-state coordinate varies monotonically with evidence, the network has built a **coding axis that stores the accumulated count** — the hallmark of an integrator. (Whether the *top* PC is that axis is a separate question — the highest-variance direction need not be the behaviourally-read one, which we probe next.) The low participation ratio says the computation lives in only a handful of dimensions regardless.

## 4. Recurrent-Jacobian spectra & effective time constants

The one-step recurrent update `h → F(h,x)` has Jacobian `J_rec = ∂F/∂h`. Its
**eigenvalues** govern local memory: `|λ| ≈ 1` is a slow/marginal mode (the network
neither forgets nor amplifies — an integrator direction), `|λ| < 1` decays. We
convert to an **effective time constant** `τ_eff = −Δt / ln|λ|` (ms): how long a
perturbation along that mode persists. A line attractor shows up as a cluster of
`|λ|` pressed against 1, i.e. `τ_eff` far exceeding the single-unit `τ = 100 ms`.

In [ ]:
from neuralgeom.dynamics import recurrent_jacobian
dt_ms = task.dt
# sample states from mid/late trial (where accumulation has happened)
sel = np.random.default_rng(1).choice(H.shape[0], 200, replace=False)
Hs = torch.tensor(H[sel], dtype=torch.float32)
Xs = torch.tensor(batch.inputs.reshape(-1, batch.inputs.shape[-1]).numpy()[sel], dtype=torch.float32)
Jr = recurrent_jacobian(model, Hs, Xs).detach().numpy()          # (200, N, N)

mod = np.abs(np.linalg.eigvals(Jr))                              # (200, N)
lead = mod.max(1)                                                # leading modulus per state
with np.errstate(divide="ignore"):
    tau_eff = -dt_ms / np.log(np.clip(lead, 1e-9, 0.999999))     # ms

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(mod.ravel(), bins=60, color="C0"); ax[0].axvline(1.0, color="C3", ls="--")
ax[0].set(title="all J_rec eigenvalue moduli |λ|", xlabel="|λ|", ylabel="count")
ax[1].hist(tau_eff[np.isfinite(tau_eff)], bins=40, color="C2")
ax[1].axvline(100, color="C3", ls="--", label="single-unit τ = 100 ms")
ax[1].set(title="effective time constant of the leading mode", xlabel="τ_eff (ms)"); ax[1].legend()
fig.tight_layout(); display(fig); plt.close(fig)
print(f"leading |λ|: median {np.median(lead):.3f}, max {lead.max():.3f}  "
      f"→ median τ_eff {np.median(tau_eff[np.isfinite(tau_eff)]):.0f} ms (vs 100 ms single-unit)")

A pile-up of eigenvalue moduli near 1 (and `τ_eff` well beyond 100 ms) is the
signature of slow, integrator-like modes — memory that is a **circuit** property,
not a single-neuron property. (These are *local* linearisations, so report them as
a distribution over states, never one number for "the network".)

## 5. Slow points — the attractor set

A **slow point** is a state where the autonomous update barely moves the state
(`‖F(h,0) − h‖` is tiny). `find_slow_points` runs an optimiser from many initial
states down to the slow set. For an integrator we expect the slow points to trace
out a **1-D curve** (the line attractor), and the accumulated-evidence coordinate
to vary along it.

In [ ]:
from neuralgeom.dynamics import find_slow_points
x0 = torch.zeros(task.spec.input_dim)                 # autonomous (no input)
h_init = torch.tensor(H[np.random.default_rng(2).choice(H.shape[0], 60, replace=False)],
                      dtype=torch.float32)
sp = find_slow_points(model, x0, h_init, steps=400, lr=0.05, verbose=False)
sp = sp.filter(q_max=1e-3)                             # keep the genuinely slow ones
hp = np.asarray(sp.h)
proj = (hp - H.mean(0)) @ Vth[:2].T

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(np.log10(np.asarray(sp.q) + 1e-12), bins=20, color="C0")
ax[0].set(title=f"slow-point speed q  ({len(hp)} kept, q<1e-3)", xlabel="log10 q", ylabel="count")
sc = ax[1].scatter(P[::8,0], P[::8,1], s=4, c="0.8")
ax[1].scatter(proj[:,0], proj[:,1], s=40, c="C3", edgecolor="k", label="slow points")
ax[1].set(title="slow points over the state cloud (PC1–PC2)", xlabel="PC1", ylabel="PC2"); ax[1].legend()
fig.tight_layout(); display(fig); plt.close(fig)
print(f"kept {len(hp)} slow points; n_unstable (>1 unstable dir): "
      f"{getattr(sp, 'n_unstable', 'n/a')}")

If the slow points line up as an elongated 1-D set threading the state cloud,
that **is** the line attractor: a continuum of storage states the network relaxes
onto and then holds.

## 6. Readout vs input subspaces

`readout_subspace` is the direction the output layer reads; `input_subspace` is
where the stimulus enters. `subspace_alignment` (via principal angles) *measures* — it does not assume — whether the readout direction coincides with the top variance direction (PC1) or with the input axis. We report `mean_cos2` (1 = identical subspaces, 0 = orthogonal); a low value is itself informative, saying the decision is read from a direction that is not the highest-variance one.

In [ ]:
from neuralgeom.dynamics import readout_subspace, input_subspace, subspace_alignment
Qr = readout_subspace(model, k=1)                       # (N,1) output-read direction
Qi = input_subspace(model, channels=[1, 2], k=1)        # (N,1) dominant stimulus axis
pc1 = torch.tensor(Vth[0:1].T, dtype=torch.float32)     # (N,1) top PC of the state cloud
# subspace_alignment returns a dict; mean_cos2 ∈ [0,1] is 1 when the subspaces coincide
al_ro_pc = subspace_alignment(Qr, pc1)["mean_cos2"]
al_ro_in = subspace_alignment(Qr, Qi)["mean_cos2"]
fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax.bar(["readout ↔ PC1", "readout ↔ input"], [al_ro_pc, al_ro_in], color=["C2","C1"])
ax.set(title="subspace alignment (1 = identical)", ylim=(0,1.02))
fig.tight_layout(); display(fig); plt.close(fig)
print(f"alignment  readout↔PC1 = {al_ro_pc:.2f}   readout↔input = {al_ro_in:.2f}")

## 7. State-space pullback metric  `g = J_recᵀ J_rec`

Unlike decoding a scalar (which gives a rank-1, information-poor metric), the
**recurrent update** maps state space to state space, so its pullback metric
`g(h) = J_recᵀ J_rec` is a genuine **full-rank field** over states. Its
**log-volume** `½ log det g` measures how much the one-step map locally expands or
contracts volume, and its **anisotropy** `λ_max/λ_min` how direction-dependent that
is. Integrator directions (|λ|≈1, neither expanding nor contracting) coexist with
strongly contracting off-manifold directions, so we expect **high anisotropy**.

In [ ]:
from neuralgeom.dynamics import state_pullback_metric
g = state_pullback_metric(model, Hs, Xs, wrt="h").detach().numpy()   # (200, N, N)
evg = np.linalg.eigvalsh(g)                                          # ascending
evg = np.clip(evg, 1e-12, None)
logvol = 0.5 * np.log(evg).sum(1)
aniso = np.log10(evg[:, -1] / evg[:, 0])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(logvol, bins=40, color="C0"); ax[0].set(title="½ log det g  (one-step log-volume change)", xlabel="nats")
ax[1].hist(aniso, bins=40, color="C4"); ax[1].set(title="anisotropy  log10(λmax/λmin)", xlabel="log10 ratio")
fig.tight_layout(); display(fig); plt.close(fig)
print(f"median log-volume {np.median(logvol):+.2f} nats (negative ⇒ net contraction), "
      f"median anisotropy 1e{np.median(aniso):.1f}")

## 8. Bridge to the subspace lens (same network, other lens)

Finally, feed the trained network's hidden states through the **subspace lens** —
wrap them in a `Trajectory` and run the exact Grassmannian embedding + persistent
homology we used on the ring. For a *line* attractor the coding subspace does **not**
rotate (the axis is fixed while the state slides along it), so we expect **little
or no persistent H1 loop** — the subspace lens is, by design, blind to
integration along a fixed axis. Seeing that "null" here is itself informative: it
says the two lenses are complementary, and that this computation is the kind the
*pullback/dynamics* half captures and the *subspace/topology* half does not.

In [ ]:
from neuralgeom.data import Trajectory
from neuralgeom.subspace import EmbedConfig, embed_from_trajectory, compute_kinematics, KinConfig
from neuralgeom.topology.persistence import single_trial_distances, ph, top_life
from persim import plot_diagrams

traj_rnn = Trajectory.from_arrays(hidden.numpy(), dt=task.dt/1000.0, generator="trained_rnn")
emb = embed_from_trajectory(traj_rnn, 0, EmbedConfig(k=2, win=20, stride=2))
kin = compute_kinematics(emb["frames"], emb["win_times"], KinConfig())
D = single_trial_distances(traj_rnn, 0, EmbedConfig(k=2, win=20, stride=2))
dg = ph(D, maxdim=1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].plot(kin["time"], kin["speed"], color="C0")
ax[0].set(title=f"subspace speed of the trained net (k=2)\ngeodesic efficiency={kin['efficiency']:.2f}",
          xlabel="time (s)", ylabel="‖v‖")
plot_diagrams(dg, ax=ax[1]); ax[1].set_title(f"persistent homology (H1 top = {top_life(dg[1]):.2f})")
fig.tight_layout(); display(fig); plt.close(fig)
print(f"subspace H1 top persistence = {top_life(dg[1]):.2f}  "
      f"(small ⇒ no rotating coding axis, consistent with a line attractor)")

## 9. Summary — what the two lenses said about the *learned* network

Applying the full `neuralgeom` stack to a network we only *trained* (never
hand-wired):

- **Behaviour:** it solves evidence integration (psychometric curve through 0.5 at
  zero evidence) and lays the accumulated evidence out **monotonically along a
  single state-space axis**.
- **Dynamics geometry:** recurrent-Jacobian eigenvalues pile up near `|λ|=1` with
  **effective time constants ≫ the single-unit τ**, and **slow points** trace a
  roughly 1-D set — a **line attractor**, discovered, not imposed.
- **Alignment (a surfaced question, not a tidy confirmation):** the `k=1` readout direction came out *largely orthogonal* to both PC1 and the input axis (`mean_cos2 ≈ 0`). That means the behavioural read-out is **not** the highest-variance direction — plausibly it reads a lower-variance A−B *contrast* (the 3-class readout's dominant mode is the fixate-vs-respond axis, not the evidence axis). Exactly the kind of non-obvious lead the pipeline is meant to raise; worth probing by aligning the readout to the *difference* of the A/B output rows.
- **Pullback metric:** the recurrent map gives a full-rank, **highly anisotropic**
  state-space metric — slow storage directions alongside strongly contracting
  off-manifold directions.
- **Subspace lens:** little/no persistent loop — correctly, because a line
  attractor stores information by moving *along a fixed axis*, which the
  Grassmannian lens is designed to be blind to.

**The combined message.** The two lenses are complementary. The ring attractor is
a *subspace/topology* story (a rotating coding direction → an H1 loop); the
integrator is a *pullback/dynamics* story (a fixed axis with marginal modes → a
line attractor and an anisotropic metric). Having both in one library, over one
`Trajectory` contract, is what lets you ask a network *which kind of computation*
it is doing. As always these are exploratory descriptions on synthetic/trained
data, and (per the roadmap) the SPD companion + null-model inference are the next
steps toward claims that hold on data with unknown structure.